In [89]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [90]:
metadata_df = pd.read_csv('../data/metadata.csv')
features_df = pd.read_csv('../data/features.csv')

metadata_df = metadata_df.drop(columns=["Unnamed: 0"])
features_df = features_df.drop(columns=["Unnamed: 0"])

In [91]:
metadata_df

,track_name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,Comedy,Gen Hoshino,0.629239,-0.717147,0.300825,0.551843,-0.850193,-0.504111,0.758735,0.929315,-1.141854
1,Ghost - Acoustic,Ben Woodward,-0.845908,-1.889974,-1.784739,-0.078995,1.831744,-0.504097,-0.591216,-0.798681,-1.489708
2,To Begin Again,Ingrid Michaelson;ZAYN,-0.742187,-1.122667,-0.293289,-0.273827,-0.315489,-0.504115,-0.507172,-1.365679,-1.528303
3,Can't Help Falling In Love,Kina Grannis,-1.733301,-2.312987,-2.039246,-0.457309,1.774605,-0.503886,-0.428381,-1.276965,1.987857
4,Hold On,Chord Overstreet,0.295026,-0.788709,-0.282751,-0.303146,0.463409,-0.504115,-0.686290,-1.184394,-0.073343
...,...,...,...,...,...,...,...,...,...,...,...
113994,Sleep My Little Boy,Rainy Lullaby,-2.274956,-1.615652,-1.617321,-0.401507,0.977663,2.493742,-0.668431,-1.697779,0.128337
113995,Water Into Light,Rainy Lullaby,-2.263432,-2.084782,-2.000075,-0.421369,2.042258,2.648803,-0.570205,-1.693536,-1.231186
113996,Miss Perfumado,Cesária Evora,0.358411,-1.241937,-0.524135,-0.403399,1.660327,-0.504115,-0.681038,1.037314,0.341259
113997,Friends,Michael W. Smith,0.116395,-0.538241,-0.522942,-0.519731,0.198764,-0.504115,0.296495,-0.235539,0.460746


In [92]:
def get_song_vector(df, track_name, artists):
    if isinstance(artists, list):
        artists = artists[0]  # Take the first artist if it's a list
    song_row = df[(df['track_name'] == track_name) & (df['artists'] == artists)]
    if not song_row.empty:
        return song_row.drop(columns=["track_name", "artists"]).iloc[0].values.astype(float)
    else:
        return None

In [93]:
song_vector = get_song_vector(metadata_df, 'Shape of You', 'Ed Sheeran')
print(song_vector)

[ 1.48782002  0.04220877  1.00926851 -0.04210939  0.80023021 -0.5041145
 -0.63271242  1.7624554  -0.87299208]


In [94]:
def rating_to_weight(rating):
    return rating / 5.0

In [95]:
def build_user_profile(df, songs, artists, ratings):
    vectors = []
    weights = []
    
    for song, artist, rating in zip(songs, artists, ratings):
        v = get_song_vector(df, song, artist)
        w = rating_to_weight(rating)
        
        if w != 0:
            vectors.append(v)
            weights.append(w)
    
    vectors = np.vstack(vectors)
    weights = np.array(weights)
    
    user_vector = np.dot(weights, vectors) / np.sum(np.abs(weights))

    return user_vector

In [120]:
def recommend_songs(songs, artists, ratings, df=metadata_df, top_n=5):
    user_vector = build_user_profile(df, songs, artists, ratings)
    
    if user_vector is None:
        return pd.DataFrame()
    
    X = df.drop(columns=["track_name", "artists"]).values
    sims = cosine_similarity(user_vector.reshape(1, -1), X)[0]
    
    df_copy = df.copy()
    df_copy["similarity"] = sims
    df_copy = df_copy[~df_copy["track_name"].isin(songs)]
    df_copy = df_copy.drop_duplicates(subset=["track_name", "artists"])
    
    return df_copy.sort_values("similarity", ascending=False).head(top_n)[["track_name", "artists", "similarity"]].values.tolist()

In [97]:
print(metadata_df.columns.tolist())

['track_name', 'artists', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [98]:
build_user_profile(metadata_df, ['Shape of You', 'Comedy', 'Friends'], ["Ed Sheeran", 'Gen Hoshino', "Michael W. Smith"], [5, 4, 5])

array([ 0.75271623, -0.38205357,  0.25963808, -0.04298762,  0.11387153,
       -0.50411357,  0.09670395,  0.81084559, -0.47347502])

In [121]:
print(recommend_songs(['Shape of You', 'Comedy', 'Friends'], ["Ed Sheeran", 'Gen Hoshino', "Michael W. Smith"], [3, 4, 1]))

[["I'm Fine", 'Sophie Pecora', 0.9617420721881025], ['Your Number REMIX (feat. Chris Brown & Kid Ink)', 'Ayo Jay;Chris Brown;Kid Ink', 0.9589378109915059], ['The Dress', 'Dijon', 0.9475667852025867], ['Acércate', 'TINI', 0.9402618521844335], ['Butterflies', 'Fiji Blue', 0.9396277758909122]]
